# This is an exploration notebook used to analyse and understand the sqlite dataset.

**This notebook is structured by exploring loans table seperately and loan_features tabel sepreately, so that the knowledge acquired can be useful when joining tables together during transformation in dbt.**

**Import Packages**

In [264]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
import re

**Connection to data**

In [265]:
data_path = Path("../data/YHP_credit_assessment_DS.sqlite")

with sqlite3.connect(data_path) as conn:
    loans = pd.read_sql_query("SELECT * FROM loans", conn)
    features = pd.read_sql_query("SELECT * FROM loan_features", conn)


------

**Data Quality Check for loans table :**

**Column Names**

In [266]:
print(loans.columns)

Index(['loan_id', 'received_date', 'is_loss', 'loss_amount'], dtype='object')


In [267]:
# info about the loans table
loans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   loan_id        10000 non-null  object 
 1   received_date  10000 non-null  object 
 2   is_loss        10000 non-null  int64  
 3   loss_amount    9996 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 312.6+ KB


**Check date range of data**

In [268]:
# Date range of the loans table
print(f"Date range of the loans table: {loans['received_date'].min()} to {loans['received_date'].max()}")

Date range of the loans table: 2018-04-11 to 2025-12-18


**Check for nulls**

In [269]:
# Function to create a null table
def create_null_table(df):
    """Creates a table showing the number and percentage of null values for each column in the DataFrame."""
    null_table = (
        pd.DataFrame({
            "column_name": df.columns,
            "null_count": df.isna().sum().values,
            "null_percentage": (df.isna().mean() * 100).round(2).values,
        })
        .query("null_count > 0")
        .sort_values("null_count", ascending=False)
        .reset_index(drop=True)
    )

    return null_table

In [270]:
# Check for null values in the loans table
display(create_null_table(loans))

,column_name,null_count,null_percentage
0,loss_amount,4,0.04


In [271]:
# Show rows with nulls in the loans table
null_rows_loans = loans[loans.isna().any(axis=1)]
print("Null rows in loans:", len(null_rows_loans))
display(null_rows_loans)

Null rows in loans: 4


,loan_id,received_date,is_loss,loss_amount
661,LRQ-574640,2018-07-13,0,NaN
3391,LRQ-470946,2018-06-03,0,NaN
8134,LRQ-645901,2018-06-03,0,NaN
9417,LRQ-240704,2018-06-03,0,NaN


Check if is_loss is a boolean (0 and 1), and verify if 1 represents loss

In [272]:
# check if the value under is_loss are only 0 and 1
loans["is_loss"].unique()

array([0, 1])

In [273]:
# statistics of loss_amount for loans where is_loss == 0 and is_loss == 1
display(loans[loans["is_loss"] == 0]["loss_amount"].describe())
display(loans[loans["is_loss"] == 1]["loss_amount"].describe())


count      7877.000000
mean        203.398873
std        3322.711203
min      -62560.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      101341.790000
Name: loss_amount, dtype: float64

count      2119.000000
mean      31584.836597
std       36845.197777
min       -6336.100000
25%        9292.190000
50%       20188.870000
75%       40786.710000
max      440332.440000
Name: loss_amount, dtype: float64

In [274]:
# check which is_loss values have loss_amount > 0, loss_amount == 0, loss_amount < 0 more than the other

loans["loss_amount_group"] = np.select(
    [
        loans["loss_amount"] > 0,
        loans["loss_amount"] == 0,
        loans["loss_amount"] < 0
    ],
    [
        "positive",
        "zero",
        "negative"
    ],
    default="missing"
)

loss_comparison = pd.crosstab(
    loans["loss_amount_group"],
    loans["is_loss"]
)

loss_comparison.columns = [
    f"is_loss_{int(col)}"
    for col in loss_comparison.columns
]

loss_comparison["higher_count"] = loss_comparison[
    ["is_loss_0", "is_loss_1"]
].idxmax(axis=1)

loss_comparison["difference"] = (
    loss_comparison["is_loss_1"]
    - loss_comparison["is_loss_0"]
)

display(loss_comparison)

,is_loss_0,is_loss_1,higher_count,difference
loss_amount_group,,,,
missing,4,0,is_loss_0,-4
negative,184,13,is_loss_0,-171
positive,133,2099,is_loss_1,1966
zero,7560,7,is_loss_0,-7553


After further inspection, it is verified that is_loss == 0 mean lender did not experience material credit loss and tend to have negative and zero loss_amount.  Whereas, when is_loss == 1, lender experienced a material credit loss or charge-off and have positive loss_amount. Also, null values in loans table are when the lender did not experience loss (0.04% of total dataset). As the task is focudsed on credit loss and risk, it is safe to assume this null values to be 0. 

**Check duplicates**

In [275]:
# Check if loan_id is unique in the loans table
is_unique_loan_id = loans["loan_id"].is_unique
print(f"Is loan_id unique? {is_unique_loan_id}")

Is loan_id unique? True


No duplicates found, so the loan_id is truly unique in loans table.

**Check loan is_loss and when not in is_loss consistency**

In [276]:
# Check if there are any loss loans with zero or negative loss amounts
loss_with_zero_or_negative_amount = loans[
(loans["is_loss"] == 1) & (loans["loss_amount"] <= 0)
]

total_number_of_loss_loans = len(loans[loans["is_loss"] == 1])  

print(
    "Number of loss loans with zero or negative loss:",
    len(loss_with_zero_or_negative_amount)
)

print(
    "Percentage of loss loans with zero or negative loss:",
    round((len(loss_with_zero_or_negative_amount) / total_number_of_loss_loans) * 100, 2),
    "%"
)

display(loss_with_zero_or_negative_amount)

Number of loss loans with zero or negative loss: 20
Percentage of loss loans with zero or negative loss: 0.94 %


,loan_id,received_date,is_loss,loss_amount,loss_amount_group
760,LRQ-790929,2025-08-12,1,-688.31,negative
1059,LRQ-840055,2025-02-12,1,-986.26,negative
1157,LRQ-246071,2025-03-23,1,0.00,zero
1355,LRQ-504561,2025-02-26,1,0.00,zero
2010,LRQ-238261,2025-11-04,1,0.00,zero
3341,LRQ-664222,2024-04-05,1,-147.46,negative
3364,LRQ-283818,2024-12-24,1,-1295.78,negative
3401,LRQ-892291,2023-12-09,1,-1345.94,negative
3424,LRQ-802531,2025-05-20,1,-2279.67,negative
3819,LRQ-340398,2024-12-24,1,-1295.78,negative


Only 0.94 % of the data is inconsistent for when SMB is at lost. Few exact figure of loss_amount is being repeated, let's see if there any patterns.

In [277]:
repeated_loss_patterns = (
    loans[
        (loans["is_loss"] == 1) &
        (loans["loss_amount"] <= 0)
    ]
    .groupby(["received_date", "loss_amount"])
    .agg(
        loan_count=("loan_id", "count"),
        loan_ids=("loan_id", lambda x: ", ".join(x))
    )
    .reset_index()
    .query("loan_count > 1")
    .sort_values(
        ["loan_count", "received_date"],
        ascending=[False, True]
    )
)

display(repeated_loss_patterns)

,received_date,loss_amount,loan_count,loan_ids
12,2025-08-12,-688.31,3,"LRQ-790929, LRQ-827813, LRQ-636970"
1,2023-12-09,-1345.94,2,"LRQ-892291, LRQ-223433"
5,2024-12-24,-1295.78,2,"LRQ-283818, LRQ-340398"
6,2025-02-12,-986.26,2,"LRQ-840055, LRQ-725084"
10,2025-03-23,0.00,2,"LRQ-246071, LRQ-769738"


Several records have exactly same loss amount and received date. This could indicate few things:
- repeated accounting adjustments
- recoveries or reversals processed in batches
- duplicated source values
rather than isolated random error. 

The data dictionary permits occasional negative realised losses, which is why the values are not to be removed. All records will be retained for further analysis.

In [278]:
# Check if there are any not loss loans with positive loss amounts
not_loss_with_positive_amount = loans[
    (loans["is_loss"] == 0) & (loans["loss_amount"] > 0)
]

print(
    "Rows where is_loss == 0 and loss_amount > 0:",
    len(not_loss_with_positive_amount)
)

print(
    "Not loss loans with positive loss percentage:",
    round((len(not_loss_with_positive_amount) / len(loans)) * 100, 2),
    "%"
)

display(not_loss_with_positive_amount)

Rows where is_loss == 0 and loss_amount > 0: 133
Not loss loans with positive loss percentage: 1.33 %


,loan_id,received_date,is_loss,loss_amount,loss_amount_group
3,LRQ-698073,2018-09-01,0,2045.16,positive
25,LRQ-494446,2018-10-27,0,80982.48,positive
112,LRQ-273189,2019-12-21,0,25973.94,positive
127,LRQ-856948,2020-09-27,0,16560.00,positive
177,LRQ-283091,2020-09-21,0,10914.05,positive
...,...,...,...,...,...
9922,LRQ-946305,2020-11-03,0,1106.69,positive
9927,LRQ-634265,2018-12-25,0,9867.22,positive
9928,LRQ-690406,2020-03-03,0,3442.90,positive
9974,LRQ-311892,2018-08-20,0,1028.49,positive


Unlike, above situation, these anamolies, when the lender is not at loss is not acceptable per data dicitonary, so further investigation must be made and should be flagged during analytics in dbt.

In [279]:
non_loss_positive = loans[
    (loans["is_loss"] == 0) &
    (loans["loss_amount"] > 0)
]

display(round(non_loss_positive["loss_amount"].describe(), 2))

count       133.00
mean      14726.05
std       19892.41
min           4.62
25%        2488.56
50%        7820.00
75%       18180.12
max      101341.79
Name: loss_amount, dtype: float64

In [280]:
# Check the quantiles of loss_amount for not loss loans with positive loss amounts
round(non_loss_positive["loss_amount"].quantile([0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1]), 2)

0.00         4.62
0.25      2488.56
0.50      7820.00
0.75     18180.12
0.90     39120.27
0.95     72288.28
0.99     82098.88
1.00    101341.79
Name: loss_amount, dtype: float64

**Distribution Checks**

In [281]:
loans.groupby("is_loss")["loss_amount"].agg(
    count="count",
    missing=lambda x: x.isna().sum(),
    minimum="min",
    median="median",
    mean="mean",
    maximum="max",
)

,count,missing,minimum,median,mean,maximum
is_loss,,,,,,
0,7877,4,-62560.0,0.00,203.398873,101341.79
1,2119,0,-6336.1,20188.87,31584.836597,440332.44


In [282]:
# Display the percentage distribution of loss loans and not loss loans
print("The distibution of SMB lending dataset in loss and not loss categories:")
print(f"Loss loans: {round((loans['is_loss'].sum() / len(loans)) * 100, 2)}%")
print(f"Not loss loans: {round(((len(loans) - loans['is_loss'].sum()) / len(loans)) * 100, 2)}%")

The distibution of SMB lending dataset in loss and not loss categories:
Loss loans: 21.19%
Not loss loans: 78.81%


In [283]:
# Display the percentage of zero loss amount loans, positive loss amount loans and negative loss amount loans
zero_loss_amount_loans = loans[loans["loss_amount"] == 0]
positive_loss_amount_loans = loans[loans["loss_amount"] > 0]
negative_loss_amount_loans = loans[loans["loss_amount"] < 0]

print(f"Zero loss amount loans: {round((len(zero_loss_amount_loans) / len(loans)) * 100, 2)}%")
print(f"Positive loss amount loans: {round((len(positive_loss_amount_loans) / len(loans)) * 100, 2)}%")
print(f"Negative loss amount loans: {round((len(negative_loss_amount_loans) / len(loans)) * 100, 2)}%")

Zero loss amount loans: 75.67%
Positive loss amount loans: 22.32%
Negative loss amount loans: 1.97%


----

**Data Quality Check for loan_features table :**

In [284]:
print(features.columns)

Index(['loan_id', 'feat_001', 'feat_002', 'feat_003', 'feat_004',
       'feat_005_std', 'feat_006', 'feat_007_min', 'feat_008', 'feat_009_min',
       'feat_010', 'feat_011', 'feat_012_trend', 'feat_013_mean',
       'feat_005_cv', 'feat_014_min', 'feat_015', 'feat_016', 'feat_017_sum',
       'feat_018', 'feat_019_max', 'feat_020_sum', 'feat_021_mean',
       'feat_022_sum', 'feat_023_min', 'feat_024_sum', 'feat_025_mean',
       'feat_026_min', 'feat_027_mean', 'feat_028_min', 'feat_029_min',
       'feat_021_trend', 'feat_030_max', 'feat_031_min', 'feat_032_sum',
       'feat_033_max', 'feat_034_mean', 'feat_035_sum', 'feat_036_sum',
       'feat_037_max', 'feat_038_mean', 'feat_039_max', 'feat_040_sum',
       'feat_041', 'feat_042_max', 'feat_014_max', 'feat_043_mean',
       'feat_044_min', 'feat_045_sum', 'feat_046_sum', 'feat_047_mean',
       'feat_048_min', 'feat_049_mean', 'feat_050_min', 'feat_051_sum',
       'feat_052_min', 'feat_053_min', 'feat_009_sum', 'feat_054_min',

In [285]:
# info about the features table
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 100 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   loan_id         10000 non-null  object 
 1   feat_001        10000 non-null  object 
 2   feat_002        9994 non-null   float64
 3   feat_003        10000 non-null  object 
 4   feat_004        10000 non-null  float64
 5   feat_005_std    10000 non-null  float64
 6   feat_006        5842 non-null   float64
 7   feat_007_min    10000 non-null  float64
 8   feat_008        5437 non-null   float64
 9   feat_009_min    10000 non-null  float64
 10  feat_010        7163 non-null   float64
 11  feat_011        10000 non-null  float64
 12  feat_012_trend  9882 non-null   float64
 13  feat_013_mean   9870 non-null   float64
 14  feat_005_cv     10000 non-null  float64
 15  feat_014_min    10000 non-null  float64
 16  feat_015        10000 non-null  float64
 17  feat_016        9791 non-null  

**Check for nulls**

In [286]:
# Check for null values in the features table
display(create_null_table(features))

,column_name,null_count,null_percentage
0,feat_008,4563,45.63
1,feat_006,4158,41.58
2,feat_034_mean,3685,36.85
3,feat_010,2837,28.37
4,feat_018,750,7.50
5,feat_060,750,7.50
6,feat_074_max,411,4.11
7,feat_037_max,411,4.11
8,feat_084,331,3.31
9,feat_016,209,2.09


In [287]:
# Show rows with nulls in the features table
null_rows_features = features[features.isna().any(axis=1)]
print("Null rows in features:", len(null_rows_features))
display(null_rows_features)

Null rows in features: 7119


,loan_id,feat_001,feat_002,feat_003,feat_004,feat_005_std,feat_006,feat_007_min,feat_008,feat_009_min,...,feat_082_min,feat_083_min,feat_084,feat_085_max,feat_005_trend,feat_022_min,feat_086_sum,feat_087_mean,feat_088_mean,feat_055_sum
0,LRQ-235185,IND_183,9.50,ST_41,1.0,3188.270996,0.000000,0.0,NaN,0.7500,...,9998.0,998.0,71.861,1.0,-3726.5,9998.0,998.0,99998.0,9.9998,998.0
1,LRQ-499729,IND_107,5.50,ST_46,1.0,10946.144531,0.375320,126824.0,NaN,1.0000,...,18.0,37.0,62.338,9.0,3942.5,2.0,267.0,273.0,9.9998,88.0
2,LRQ-711628,IND_055,3.50,ST_07,1.0,4439.699219,0.354667,0.0,138.0,1.0000,...,9995.0,63.0,60.606,7.0,4336.0,19.0,48.0,1271.0,9.9998,996.0
4,LRQ-985316,IND_107,7.25,ST_19,1.0,14674.486328,NaN,0.0,28.0,0.7727,...,327.0,52.0,64.502,9.0,8992.5,173.0,88.0,638.0,9.9998,80.0
5,LRQ-331394,IND_018,7.50,ST_47,1.0,9593.037109,0.494724,0.0,166.0,1.0000,...,218.0,48.0,81.602,4.0,-11749.0,1320.0,192.0,10664.0,9.9998,138.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,LRQ-967752,IND_048,0.80,ST_07,1.0,3037.941895,NaN,0.0,52.0,0.6667,...,9995.0,75.0,19.048,8.0,-3607.0,-2.0,168.0,6882.5,9.9998,1088.0
9995,LRQ-350203,IND_092,4.16,ST_46,1.0,17484.716797,NaN,0.0,NaN,0.8182,...,9995.0,0.0,54.113,7.0,13767.0,-20.0,1016.0,281.5,9.9998,1992.0
9997,LRQ-973820,IND_092,1.00,ST_07,1.0,106356.828125,0.000000,0.0,46.0,0.7333,...,9997.0,997.0,56.061,1.0,111818.5,9997.0,203.0,99997.0,9.9997,108.0
9998,LRQ-116150,IND_165,8.66,ST_32,1.0,13528.140625,NaN,0.0,NaN,0.6875,...,200.0,70.0,38.961,6.0,-15887.0,5.0,180.0,1273.0,9.9998,135.0


feat_001, feat_003 and feat_004 seem to be categorical, so further analysis to confirm if it is categorical, and see any patterns/trends.

**Check loan id is unique in feature**

In [288]:
# Check if loan_id is unique in the features table
unique_feature_loan_id = features["loan_id"].is_unique
print(f"Is loan_id unique? {unique_feature_loan_id}")

Is loan_id unique? True


As there way too many columns to analyse and all of them are anonymised (with no data dictionary), firstly, I will try to examine them by grouping with with feat only, feat min, feat max, feat mean, feat sum, feat trend, feat cv and feat std.

In [289]:
suffixes = []

for col in features.columns:
    match = re.match(r"^feat_\d+(?:_(.+))?$", col)

    if match:
        suffix = match.group(1)

        # Columns such as feat_001 have no suffix
        suffixes.append(suffix if suffix is not None else "feature_only")

unique_suffixes = sorted(set(suffixes))

print(unique_suffixes)

['cv', 'feature_only', 'max', 'mean', 'min', 'std', 'sum', 'trend']


In [290]:
suffix_summary = (
    pd.Series(suffixes, name="suffix")
    .value_counts()
    .rename_axis("suffix")
    .reset_index(name="column_count")
)

display(suffix_summary)

,suffix,column_count
0,min,27
1,mean,18
2,sum,16
3,max,16
4,feature_only,14
5,trend,4
6,std,2
7,cv,2


**Group feat by Suffix**

In [291]:
feature_groups = {
    "feature_only": features.filter(regex=r"^feat_\d+$").columns.tolist(),
    "min": features.filter(regex=r"^feat_\d+_min$").columns.tolist(),
    "max": features.filter(regex=r"^feat_\d+_max$").columns.tolist(),
    "mean": features.filter(regex=r"^feat_\d+_mean$").columns.tolist(),
    "sum": features.filter(regex=r"^feat_\d+_sum$").columns.tolist(),
    "trend": features.filter(regex=r"^feat_\d+_trend$").columns.tolist(),
    "cv": features.filter(regex=r"^feat_\d+_cv$").columns.tolist(),
    "std": features.filter(regex=r"^feat_\d+_std$").columns.tolist(),
}

**Feat**

In [292]:
features[feature_groups["feature_only"]].head(10)

,feat_001,feat_002,feat_003,feat_004,feat_006,feat_008,feat_010,feat_011,feat_015,feat_016,feat_018,feat_041,feat_060,feat_084
0,IND_183,9.50,ST_41,1.0,0.000000,NaN,676.000000,0.032163,0.0,0.0,NaN,B,NaN,71.861
1,IND_107,5.50,ST_46,1.0,0.375320,NaN,1208.579956,0.035547,0.0,0.0,0.773881,A,84420.0,62.338
2,IND_055,3.50,ST_07,1.0,0.354667,138.0,197.729996,0.089480,0.0,0.0,0.558400,C,3750.0,60.606
3,IND_165,17.00,ST_41,0.0,1.203733,23.0,90.000000,0.205759,0.0,3410.0,0.536156,A,8740.0,37.662
4,IND_107,7.25,ST_19,1.0,NaN,28.0,2487.000000,0.000008,0.0,0.0,0.629809,A,18350.0,64.502
5,IND_018,7.50,ST_47,1.0,0.494724,166.0,171.110001,0.060796,1.0,0.0,0.430889,A,22500.0,81.602
6,IND_216,10.08,ST_32,1.0,0.726394,78.0,164.000000,0.216247,0.0,832.0,0.492592,C,204100.0,46.537
7,IND_161,2.08,ST_22,1.0,NaN,39.0,500.000000,0.452692,0.0,0.0,0.561798,B,13350.0,54.545
8,IND_064,1.50,ST_33,1.0,NaN,62.0,232.250000,0.128497,1.0,10363.0,NaN,A,NaN,44.589
9,IND_064,12.65,ST_37,1.0,NaN,NaN,2.000000,0.014695,1.0,9976.0,0.884576,A,5900.0,52.381


In [293]:
# Check the info of the selected feat only columns
features[feature_groups["feature_only"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   feat_001  10000 non-null  object 
 1   feat_002  9994 non-null   float64
 2   feat_003  10000 non-null  object 
 3   feat_004  10000 non-null  float64
 4   feat_006  5842 non-null   float64
 5   feat_008  5437 non-null   float64
 6   feat_010  7163 non-null   float64
 7   feat_011  10000 non-null  float64
 8   feat_015  10000 non-null  float64
 9   feat_016  9791 non-null   float64
 10  feat_018  9250 non-null   float64
 11  feat_041  10000 non-null  object 
 12  feat_060  9250 non-null   float64
 13  feat_084  9669 non-null   float64
dtypes: float64(11), object(3)
memory usage: 1.1+ MB


In [294]:
# Check the percentage of missing values in the feat only columns
features[feature_groups["feature_only"]].isna().mean()*100

feat_001     0.00
feat_002     0.06
feat_003     0.00
feat_004     0.00
feat_006    41.58
feat_008    45.63
feat_010    28.37
feat_011     0.00
feat_015     0.00
feat_016     2.09
feat_018     7.50
feat_041     0.00
feat_060     7.50
feat_084     3.31
dtype: float64

In [295]:
# Ranking each feat only based on the percentage of missing values in descending order
display(create_null_table(features[feature_groups["feature_only"]]))

,column_name,null_count,null_percentage
0,feat_008,4563,45.63
1,feat_006,4158,41.58
2,feat_010,2837,28.37
3,feat_018,750,7.50
4,feat_060,750,7.50
5,feat_084,331,3.31
6,feat_016,209,2.09
7,feat_002,6,0.06


In [296]:
# Display feature only columnns with no missing values
display(
    features[feature_groups["feature_only"]].columns[
        features[feature_groups["feature_only"]].isna().sum() == 0
        ].tolist()
)

['feat_001', 'feat_003', 'feat_004', 'feat_011', 'feat_015', 'feat_041']

**Check feat_001**

In [297]:
def summarise_category(
    dataframe,
    column_name,
    top_n=20,
    include_missing=False
):
    """
    Summarise the values in a categorical column.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        DataFrame containing the column.
    column_name : str
        Name of the column to summarise.
    top_n : int, default=20
        Number of most frequent categories to display.
    include_missing : bool, default=False
        Whether to include missing feature values as a separate group.

    Returns
    -------
    pandas.DataFrame
        Category counts and percentages.
    """

    print("Column:", column_name)
    print("Data type:", dataframe[column_name].dtype)
    print(
        "Unique categories:",
        dataframe[column_name].nunique(dropna=False)
    )

    summary = (
        dataframe[column_name]
        .value_counts(dropna=False) 
        .rename_axis(column_name)
        .reset_index(name="loan_count")
    )

    summary["loan_percentage"] = (
        summary["loan_count"] / len(dataframe) * 100
    ).round(2)

    display(summary.head(top_n))

    return summary

In [298]:
summarise_category(features, "feat_001")

Column: feat_001
Data type: object
Unique categories: 232


,feat_001,loan_count,loan_percentage
0,IND_183,965,9.65
1,IND_064,882,8.82
2,IND_048,511,5.11
3,IND_019,411,4.11
4,IND_020,407,4.07
5,IND_012,324,3.24
6,IND_092,272,2.72
7,IND_018,240,2.40
8,IND_107,230,2.30
9,IND_191,226,2.26


,feat_001,loan_count,loan_percentage
0,IND_183,965,9.65
1,IND_064,882,8.82
2,IND_048,511,5.11
3,IND_019,411,4.11
4,IND_020,407,4.07
...,...,...,...
227,IND_144,1,0.01
228,IND_176,1,0.01
229,IND_177,1,0.01
230,IND_172,1,0.01


In [299]:
def analyse_feature_by_loss(
    loans,
    features,
    feature_name,
    top_n=20,
    include_missing=False
):
    """
    Analyse loan performance by a selected feature.

    Parameters
    ----------
    loans : pandas.DataFrame
        Must contain loan_id, is_loss, and loss_amount.
    features : pandas.DataFrame
        Must contain loan_id and the selected feature.
    feature_name : str
        Name of the feature to analyse.
    top_n : int, default=20
        Number of feature groups to display.
    include_missing : bool, default=False
        Whether to include missing feature values as a separate group.

    Returns
    -------
    pandas.DataFrame
        Full feature-level loss analysis.
    """

    feature_analysis = (
        loans[["loan_id", "is_loss", "loss_amount"]]
        .merge(
            features[["loan_id", feature_name]],
            on="loan_id",
            how="left",
            validate="one_to_one"
        )
        .groupby(
            feature_name,
            dropna=not include_missing
        )
        .agg(
            loan_count=("loan_id", "count"),
            loss_count=("is_loss", "sum"),
            loss_rate=("is_loss", "mean"),
            average_loss_amount=("loss_amount", "mean")
        )
        .reset_index()
    )

    feature_analysis["loss_rate_pct"] = (
        feature_analysis["loss_rate"] * 100
    )

    feature_analysis = feature_analysis.sort_values(
        "loan_count",
        ascending=False
    )

    display(
        feature_analysis
        .head(top_n)
        .round(2)
    )

    return feature_analysis

In [300]:
analyse_feature_by_loss(loans, features, "feat_001", top_n=20, include_missing=True)

,feat_001,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
182,IND_183,965,254,0.26,8304.73,26.32
63,IND_064,882,178,0.20,5787.23,20.18
47,IND_048,511,92,0.18,5490.01,18.00
18,IND_019,411,129,0.31,16722.03,31.39
19,IND_020,407,37,0.09,3047.84,9.09
11,IND_012,324,68,0.21,5108.61,20.99
91,IND_092,272,59,0.22,5116.04,21.69
17,IND_018,240,57,0.24,5777.17,23.75
106,IND_107,230,47,0.20,7190.02,20.43
190,IND_191,226,52,0.23,6883.35,23.01


,feat_001,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
182,IND_183,965,254,0.263212,8304.725264,26.321244
63,IND_064,882,178,0.201814,5787.225057,20.181406
47,IND_048,511,92,0.180039,5490.008611,18.003914
18,IND_019,411,129,0.313869,16722.031849,31.386861
19,IND_020,407,37,0.090909,3047.840811,9.090909
...,...,...,...,...,...,...
191,IND_192,1,0,0.000000,0.000000,0.000000
175,IND_176,1,0,0.000000,0.000000,0.000000
173,IND_174,1,1,1.000000,21014.240000,100.000000
171,IND_172,1,1,1.000000,64778.650000,100.000000


**Feat 003**

In [314]:
summarise_category(features, "feat_003", top_n=20, include_missing=True)

Column: feat_003
Data type: object
Unique categories: 55


,feat_003,loan_count,loan_percentage
0,ST_32,1346,13.46
1,ST_46,1330,13.30
2,ST_07,982,9.82
3,ST_06,570,5.70
4,ST_50,475,4.75
5,ST_08,444,4.44
6,ST_39,329,3.29
7,ST_41,301,3.01
8,ST_03,296,2.96
9,ST_27,267,2.67


,feat_003,loan_count,loan_percentage
0,ST_32,1346,13.46
1,ST_46,1330,13.30
2,ST_07,982,9.82
3,ST_06,570,5.70
4,ST_50,475,4.75
5,ST_08,444,4.44
6,ST_39,329,3.29
7,ST_41,301,3.01
8,ST_03,296,2.96
9,ST_27,267,2.67


In [325]:
analyse_feature_by_loss(loans, features, "feat_003", top_n=20, include_missing=True)

,feat_003,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
31,ST_32,1346,309,0.23,9559.14,22.96
45,ST_46,1330,269,0.20,5764.20,20.23
6,ST_07,982,203,0.21,7455.16,20.67
5,ST_06,570,112,0.20,6622.77,19.65
49,ST_50,475,122,0.26,7092.91,25.68
7,ST_08,444,56,0.13,2874.61,12.61
38,ST_39,329,68,0.21,7531.62,20.67
40,ST_41,301,70,0.23,6388.35,23.26
2,ST_03,296,71,0.24,7062.24,23.99
26,ST_27,267,51,0.19,4915.61,19.10


,feat_003,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
31,ST_32,1346,309,0.229569,9559.141924,22.956909
45,ST_46,1330,269,0.202256,5764.195699,20.225564
6,ST_07,982,203,0.206721,7455.162699,20.672098
5,ST_06,570,112,0.196491,6622.765439,19.649123
49,ST_50,475,122,0.256842,7092.908316,25.684211
7,ST_08,444,56,0.126126,2874.611757,12.612613
38,ST_39,329,68,0.206687,7531.620334,20.668693
40,ST_41,301,70,0.232558,6388.346678,23.255814
2,ST_03,296,71,0.239865,7062.241858,23.986486
26,ST_27,267,51,0.191011,4915.613521,19.101124


**Feat 004**

In [318]:
summarise_category(features, "feat_004", include_missing=True)

Column: feat_004
Data type: float64
Unique categories: 2


,feat_004,loan_count,loan_percentage
0,1.0,6362,63.62
1,0.0,3638,36.38


,feat_004,loan_count,loan_percentage
0,1.0,6362,63.62
1,0.0,3638,36.38


In [320]:
analyse_feature_by_loss(loans, features, "feat_004", top_n=20, include_missing=True)

,feat_004,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
1,1.0,6362,1586,0.25,7994.30,24.93
0,0.0,3638,533,0.15,4865.19,14.65


,feat_004,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
1,1.0,6362,1586,0.249293,7994.299341,24.929268
0,0.0,3638,533,0.146509,4865.188936,14.650907


**Feat 15**

In [321]:
summarise_category(features, "feat_015", include_missing=True)

Column: feat_015
Data type: float64
Unique categories: 2


,feat_015,loan_count,loan_percentage
0,0.0,5860,58.6
1,1.0,4140,41.4


,feat_015,loan_count,loan_percentage
0,0.0,5860,58.6
1,1.0,4140,41.4


In [322]:
analyse_feature_by_loss(loans, features, "feat_015", top_n=20, include_missing=False)

,feat_015,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
0,0.0,5860,1367,0.23,7663.78,23.33
1,1.0,4140,752,0.18,5711.00,18.16


,feat_015,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
0,0.0,5860,1367,0.233276,7663.782312,23.327645
1,1.0,4140,752,0.181643,5710.995484,18.164251


**Feat 41**

In [323]:
summarise_category(features, "feat_041", include_missing=True)

Column: feat_041
Data type: object
Unique categories: 4


,feat_041,loan_count,loan_percentage
0,A,5307,53.07
1,B,3165,31.65
2,C,1488,14.88
3,D,40,0.40


,feat_041,loan_count,loan_percentage
0,A,5307,53.07
1,B,3165,31.65
2,C,1488,14.88
3,D,40,0.40


In [324]:
analyse_feature_by_loss(loans, features, "feat_041", top_n=20, include_missing=False)

,feat_041,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
0,A,5307,1197,0.23,6856.63,22.56
1,B,3165,616,0.19,7839.58,19.46
2,C,1488,299,0.20,4910.67,20.09
3,D,40,7,0.18,1333.94,17.50


,feat_041,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
0,A,5307,1197,0.225551,6856.629126,22.555116
1,B,3165,616,0.194629,7839.575822,19.462875
2,C,1488,299,0.200941,4910.666082,20.094086
3,D,40,7,0.175000,1333.941250,17.500000


----

In [308]:
missingness_results = []

for col in features.columns:
    if col == "loan_id" or features[col].isna().sum() == 0:
        continue

    temp = (
        features[["loan_id", col]]
        .merge(
            loans[["loan_id", "is_loss", "loss_amount"]],
            on="loan_id",
            how="inner"
        )
    )

    temp["is_missing"] = temp[col].isna()

    summary = (
        temp.groupby("is_missing")
        .agg(
            loan_count=("loan_id", "count"),
            loss_rate=("is_loss", "mean"),
            average_loss_amount=("loss_amount", "mean")
        )
    )

    missing_loss_rate = summary.loc[True, "loss_rate"]
    present_loss_rate = summary.loc[False, "loss_rate"]

    missingness_results.append({
        "feature": col,
        "missing_count": temp["is_missing"].sum(),
        "missing_pct": temp["is_missing"].mean() * 100,
        "loss_rate_when_missing": missing_loss_rate * 100,
        "loss_rate_when_present": present_loss_rate * 100,
        "loss_rate_difference": (missing_loss_rate - present_loss_rate) * 100
    })

missingness_analysis = (
    pd.DataFrame(missingness_results)
    .sort_values(
        "loss_rate_difference",
        key=abs,
        ascending=False
    )
)

display(round(missingness_analysis, 2))

,feature,missing_count,missing_pct,loss_rate_when_missing,loss_rate_when_present,loss_rate_difference
0,feat_002,6,0.06,33.33,21.18,12.15
6,feat_016,209,2.09,28.71,21.03,7.68
1,feat_006,4158,41.58,24.77,18.64,6.13
3,feat_010,2837,28.37,25.31,19.56,5.75
5,feat_013_mean,130,1.30,16.15,21.26,-5.10
8,feat_019_max,130,1.30,16.15,21.26,-5.10
11,feat_029_min,130,1.30,16.15,21.26,-5.10
10,feat_028_min,130,1.30,16.15,21.26,-5.10
9,feat_026_min,130,1.30,16.15,21.26,-5.10
40,feat_087_mean,130,1.30,16.15,21.26,-5.10


In [309]:
data = features.merge(
    loans[["loan_id", "received_date", "is_loss"]],
    on="loan_id",
    how="inner"
)

data["received_year"] = pd.to_datetime(
    data["received_date"]
).dt.year

high_missing_features = [
    "feat_008",
    "feat_006",
    "feat_034_mean",
    "feat_010"
]

missing_by_year = (
    data.groupby("received_year")[high_missing_features]
    .apply(lambda frame: frame.isna().mean() * 100)
)

display(missing_by_year.round(2))

,feat_008,feat_006,feat_034_mean,feat_010
received_year,,,,
2018,63.67,44.68,43.42,33.29
2019,63.28,36.07,43.52,29.48
2020,41.19,31.14,36.35,24.19
2021,38.82,30.26,36.84,20.39
2022,33.69,43.52,41.98,28.66
2023,44.37,44.70,38.50,32.79
2024,46.32,42.87,30.55,25.64
2025,42.97,41.34,28.11,23.63
